In [ ]:
🛠️ How To Compute (Practically in R)
If you have a sample from the distribution (common in Bayesian posteriors or from simulation), you can estimate it as follows:

Method 1: Using hdrcde package (recommended)
r
Copy
Edit
install.packages("hdrcde")  # if not already installed
library(hdrcde)

# Suppose `x` is your sample:
hdr_result <- hdr(x, prob = 10)

# This gives the HDR intervals for 10% mass
print(hdr_result)
Method 2: Manual approach
r
Copy
Edit
# Assuming x is your sample
density_est <- density(x)
dx <- data.frame(x = density_est$x, y = density_est$y)
dx <- dx[order(-dx$y), ]

# Cumulative sum of areas between points
dx$area <- c(0, cumsum(0.5 * diff(dx$x) * (head(dx$y, -1) + tail(dx$y, -1))))
dx <- dx[order(dx$y, decreasing = TRUE), ]
dx$cumprob <- cumsum(c(0, diff(dx$x) * head(dx$y, -1)))  # approximation

# Keep values until we hit 10% of probability
threshold_idx <- which(cumsum(dx$area) >= 0.10)[1]
density_threshold <- dx$y[threshold_idx]

# Now identify the region(s)
hdr_region <- dx$x[dx$y >= density_threshold]
This gives you the points with highest density that contain ~10% of the mass.

📌 Summary
You're describing a Highest Density Region (HDR):

The interval of most frequently occurring values that sum to a desired probability (10%).

It includes the mode by definition.

No point outside the interval has a higher density than any point inside.

It may be non-contiguous if the distribution is multimodal.

Let me know if you'd like help implementing this in R with your actual data or samples.








You said:
Thanks! That's actually super helpful, I'll see how it works and let you know if I need more help!


ChatGPT said:
You're very welcome — I'm glad it was helpful! 🙌

Give it a try, and definitely reach out if you run into any issues or want to dig deeper (e.g., working with multimodal cases, visualizing HDRs, or applying it to 2D/3D distributions). Good luck with your analysis!







You said:
For skewnormal - is it possible to do it with sn package?


ChatGPT said:
Yes, the sn package in R is well-suited for working with skew-normal and related distributions (like skew-t), and you can use it to compute the Highest Density Region (HDR) — though it doesn’t offer a built-in function for HDR directly. You’ll just need to:

Evaluate the skew-normal density function.

Numerically find the density threshold that gives you the top 10% of the distribution.

Extract the regions where the density is above that threshold.

✅ Step-by-Step HDR for Skew-Normal Using sn Package
1. Load the package and define parameters
r
Copy
Edit
library(sn)

# Define skew-normal parameters
xi <- 0      # location
omega <- 1   # scale
alpha <- 5   # shape (positive = right-skew)

# Define the PDF
f <- function(x) dsn(x, xi, omega, alpha)
2. Create a fine grid over the range of interest
r
Copy
Edit
x_vals <- seq(xi - 5*omega, xi + 5*omega, length.out = 10000)
densities <- f(x_vals)
3. Sort by density and compute cumulative probability
r
Copy
Edit
# Create data frame
df <- data.frame(x = x_vals, f = densities)

# Sort by decreasing density
df_sorted <- df[order(-df$f), ]

# Approximate cumulative probability using trapezoidal rule
dx <- diff(df_sorted$x)
avg_heights <- (head(df_sorted$f, -1) + tail(df_sorted$f, -1)) / 2
cum_prob <- cumsum(dx * avg_heights)

# Find threshold where cumulative prob hits 10%
threshold_index <- which(cum_prob >= 0.10)[1]
f_threshold <- df_sorted$f[threshold_index]
4. Extract HDR region(s)
r
Copy
Edit
# Identify where density is above threshold
hdr_df <- df[df$f >= f_threshold, ]

# Optional: identify contiguous intervals (if multimodal)
library(dplyr)
hdr_df <- hdr_df %>%
  mutate(group = cumsum(c(1, diff(x) > (mean(diff(x_vals)) * 1.5))))

# Summarize intervals
hdr_intervals <- hdr_df %>%
  group_by(group) %>%
  summarise(start = min(x), end = max(x))
5. Plot (optional)
r
Copy
Edit
plot(x_vals, densities, type = "l", main = "HDR for Skew-Normal (10%)")
abline(h = f_threshold, col = "red", lty = 2)
for (i in 1:nrow(hdr_intervals)) {
  rect(hdr_intervals$start[i], 0, hdr_intervals$end[i], f_threshold,
       col = rgb(1, 0, 0, 0.2), border = NA)
}
🔁 Alternative: Sample-Based HDR
If you prefer simulation:

r
Copy
Edit
samples <- rsn(1e6, xi, omega, alpha)
library(hdrcde)
hdr(samples, prob = 10)
That gives a quick and effective HDR estimate based on random draws, which is often more robust in practice — especially for complicated shapes.

✅ Summary
The sn package gives you the tools to evaluate and sample from skew-normal distributions.

You can compute the HDR manually using sorted density values and numerical integration.

Alternatively, use the sample-based method with hdrcde::hdr() for simplicity and flexibility.

Let me know if you'd like help adapting this for multivariate skew-normal (msn) or skew-t distributions!